# Hour 4 — Bonus: Graphs & Visualizations

Recap: Hour 1 covered auth/projects/companies/users, Hour 2 covered file areas/folders/files, Hour 3 covered
tasks/forms/work packages. This bonus hour doesn't add new `dalux_build` methods — instead it takes the
`to_dataframe=True` results you already know how to fetch and turns them into charts with `matplotlib`.

**By the end of this hour you will be able to:**

- Turn any `to_dataframe=True` result into a chart with a few lines of `matplotlib`
- Build bar charts from companies/users, work packages, and the folder tree from Hour 2
- Handle loosely-typed resources (tasks, work packages) defensively before charting them
- Combine several charts into one dashboard figure and save it to a PNG


## 0. Reconnect

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Works whether Jupyter was launched from the repo root or from tutorials/
for candidate in (Path(".env"), Path("../.env")):
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()  # fall back to variables already exported in the shell

import os
assert os.getenv("DALUX_API_KEY"), "DALUX_API_KEY not found — copy .env.example to .env and fill it in"
assert os.getenv("DALUX_BASE_URL"), "DALUX_BASE_URL not found — copy .env.example to .env and fill it in"
print("DALUX_BASE_URL:", os.getenv("DALUX_BASE_URL"))

In [ ]:
from dalux_build import create_client

dalux = create_client()
dalux

In [ ]:
# Pick the first project in your account to work with for the rest of this notebook.
# Swap this for dalux.projects.get_project_by_name("Your Project Name") if you want a specific one.
projects_response = dalux.projects.list_projects()

PROJECT_ID = projects_response[0].project_id
print("Using project:", projects_response[0].project_name, f"({PROJECT_ID})")

dalux.set_default_project(PROJECT_ID)

In [ ]:
# Several charts/jobs in this hour need a file_area_id too — reuse the Hour 2 pattern.
file_areas_response = dalux.file_areas.get_file_areas()

FILE_AREA_ID = file_areas_response[0].file_area_id
FILE_AREA_NAME = file_areas_response[0].file_area_name
print("Using file area:", FILE_AREA_NAME, f"({FILE_AREA_ID})")

dalux.set_default_file_area(FILE_AREA_ID)

## 1. Plotting setup

`matplotlib` isn't a dependency of `dalux_build` itself (it's a plotting choice, not an API concern), so add it to
this project the same way you added `tabulate` in Hour 1.

In [ ]:
!uv add matplotlib -qU

import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110  # crisper inline charts

## 2. Bar chart — users per company

Reuse the `companies` and `users` DataFrames from Hour 1: group users by `companyId`, join in the company name,
and chart the top 15 companies by headcount. A horizontal bar chart reads better than a vertical one once you have
this many categories and long labels.

In [ ]:
companies_df = dalux.companies.list_project_companies(to_dataframe=True)
users_df = dalux.users.list_project_users(to_dataframe=True)

users_per_company = (
    users_df.groupby("companyId")
    .size()
    .rename("userCount")
    .reset_index()
    .merge(companies_df[["companyId", "name"]], on="companyId", how="left")
    .sort_values("userCount", ascending=False)
    .head(15)
)
users_per_company

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(users_per_company["name"], users_per_company["userCount"])
ax.invert_yaxis()  # largest company on top
ax.set_xlabel("Users")
ax.set_title("Users per company (top 15)")
fig.tight_layout()

## 3. Bar chart — work packages per company

`WorkPackage` is a loosely-typed model like `Task` (`extra="allow"`), but `workpackage_id`, `company_id` and
`name` are always present, which is enough for a clean breakdown by company.

In [ ]:
work_packages_df = dalux.work_packages.list_work_packages(to_dataframe=True)

if work_packages_df.empty or work_packages_df["companyId"].isnull().all():
    print("No work packages on this project — skipping this chart")
else:
    wp_per_company = (
        work_packages_df.groupby("companyId")
        .size()
        .rename("workPackageCount")
        .reset_index()
        .merge(companies_df[["companyId", "name"]], on="companyId", how="left")
        .sort_values("workPackageCount", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(wp_per_company["name"], wp_per_company["workPackageCount"])
    ax.set_ylabel("Work packages")
    ax.set_title("Work packages per company")
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()

## 4. Bar chart — files per top-level folder

Reuse the folder tree built with `dalux.folders.get_file_area_tree()` in Hour 2 instead of calling the API again —
each node already carries its own `files` list, so counting them is just a dict comprehension.

In [ ]:
tree = dalux.folders.get_file_area_tree(files_api=dalux.files, verbose=True)

def label(node):
    return node["name"] if node["path"] == "" else node["path"].split("/")[-1]

folder_counts = {label(child): len(child["files"]) for child in tree["children"]}

# Check if the values are all zero, which would indicate no files in any top-level folder
if not folder_counts or all(count == 0 for count in folder_counts.values()):
    print("No top-level folders in this file area — skipping this chart")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(folder_counts.keys(), folder_counts.values())
    ax.set_ylabel("Files")
    ax.set_title(f"Files per top-level folder — {FILE_AREA_NAME}")
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()

## 5. Pie chart — tasks by status

`Task` only guarantees `task_id` (`extra="allow"`) — whether a `status` column shows up in the DataFrame depends
on the task types on your project, so check for it before charting instead of assuming it's there.

In [ ]:
tasks_df = dalux.tasks.get_project_tasks(to_dataframe=True)

if tasks_df.empty or "type::name" not in tasks_df.columns:
    print("No tasks with a 'type::name' field on this project — skipping this chart")
else:
    status_counts = tasks_df["type::name"].value_counts()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(status_counts.values, labels=status_counts.index, autopct="%1.0f%%", startangle=90)
    ax.set_title("Tasks by type::name")

## 6. A small dashboard — combining charts with `subplots`

Reuse the data computed above and lay four charts out in a 2×2 grid — this is the kind of one-page project
overview you'd screenshot into a status email.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].barh(users_per_company["name"], users_per_company["userCount"])
axes[0, 0].invert_yaxis()
axes[0, 0].set_title("Users per company")

if not work_packages_df.empty:
    axes[0, 1].bar(wp_per_company["name"], wp_per_company["workPackageCount"])
    axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].set_title("Work packages per company")

axes[1, 0].bar(folder_counts.keys(), folder_counts.values())
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].set_title("Files per top-level folder")

if not tasks_df.empty and "status" in tasks_df.columns:
    axes[1, 1].pie(status_counts.values, labels=status_counts.index, autopct="%1.0f%%")
axes[1, 1].set_title("Tasks by status")

fig.suptitle(f"Project overview — {projects_response[0].project_name}", fontsize=14)
fig.tight_layout()

## 7. Saving a chart to a file

In [ ]:
fig.savefig("project_overview.png", dpi=150, bbox_inches="tight")
print("Saved project_overview.png")

## Recap & what's next

You can now turn any `to_dataframe=True` result into a chart, build category breakdowns (companies, work
packages, task status), reuse the Hour 2 folder tree for file-count charts, combine several charts into one
dashboard figure, and export a chart to PNG.

**Next up — Hour 5:** the embedded `dalux.webhook_server` — poll a file area on a schedule and get an outbound
callback when files change, or when a set of models hasn't been touched in a week.
